### Computational Guided Inquiry for Modeling Earth's Climate (Neshyba, Deloya Garcia, and Eklof, 2026)

# ClimateStats

## Overview
The idea of this module is to develop your skill and insights into the statistical analysis of weather records that lead to climate. The weather data we'll be accessing is archived at the NOAA website, https://gml.noaa.gov/dv/data/index.php?category=Meteorology&frequency=Hourly%2BAverages. We'll focus (as the name suggests) on statistics of _hourly_ measurements of key weather variables, including temperature, wind speed, wind direction. 

The main computing resource we'll be using to look at these data is the data management tool *pandas* to organize data and metadata associated with carbon emissions over time. We'll be using *pandas* a little more here than previously, in that we'll be using it to search for flags that indicate missing or bad data. 

In term of climate literacy, the important lesson here is the idea that climate science is a statistical science, specifically the statistics of weather, and therefore we would expect that statistical evidence of climate change would be most pronounced in polar regions.   


## Learning goals
1. I can find and interpret metadata for NOAA weather records.
1. I can use *pandas* to read in tables of data as dataframes, implement quality-control measures, and combine dataframes.
1. I can compute a probability density from a time series of measurements.

In [ ]:
# Get some resources
import numpy as np
import matplotlib.pyplot as plt; plt.rc("figure", figsize=(8,6))
import pandas as pd

### Interpreting metadata
It's often useful to inspect metadata associated with a data set before diving in. In the cell below, answer the prompted questions.

1. Go to the NOAA website for hourly averages, https://gml.noaa.gov/dv/data/index.php?category=Meteorology&frequency=Hourly%2BAverages, and enter "BRW" in the dropdown for station codes. What does "BRW" stand for?
YOUR ANSWER HERE

2. If you go to https://gml.noaa.gov/data/dataset.php?item=brw-met-hourly-1973, you'll be able to find the units for temperature that the archive is reporting in (you can search for "temperature"). What are the units of temperature in the archive?
YOUR ANSWER HERE

3. In same area of that document, you'll also be able to find the flag for missing temperature. It's a value that we'll need below to filter out bad data. What's the value for missing data?
YOUR ANSWER HERE

### Loading (and plotting) time series of hourly temperature data, using Pandas
Next, we'll collect some data for a recent year from the NOAA website, as a *pandas dataframe*. Execute the cell below and have a look at the output.

In [ ]:
df2024 = pd.read_csv('https://gml.noaa.gov/aftp/data/meteorology/in-situ/brw/met_brw_insitu_1_obop_hour_2024.txt',
                        delimiter=r"\s+",header=None, 
                        usecols=[0,1,2,3,4,5,6,9],
                        names=['station','year','month','day','hour','winddirection','windspeed','temperature']) 

# Print some information about the dataframe
display(df2024)

# Plotting the temperatures as a function of month
plt.figure()
plt.plot(df2024['month'], df2024['temperature'], 'kx', label='2024')
plt.xlabel('month')
plt.ylabel('Temperature (C)')
plt.title('Hourly temperatures, by month, for year 2024')
plt.grid()
plt.legend()

### Quality control
Take a close look at this graph. You'll probably notice some absurdly low "temperatures", close to -1000 degrees! Don't be alarmed -- this is not https://en.wikipedia.org/wiki/The_Day_After_Tomorrow. Those values are flags that mark the data as being bad or missing. This is a quality-control issue.

The code below directs *pandas* to look at each row of the dataframe, and if the temperature is -999.9, it records the index of that row, in a list called "badindices". The next line of code is a very cool Pandas functionality: it drops (gets rid of) those indices from the dataframe! Afterward, we re-plot the data, without the bad data.

By the way, if you execute this cell twice, you'll see that the second time it says it isn't dropping any points -- because (as I'm sure you've guessed) it got all the bad data the first time.

In [ ]:
# Find bad temperatures
badindices = df2024[ df2024['temperature'] == -999.9 ].index
print('I am dropping this many missing data points: ', len(badindices))
df2024.drop(badindices,inplace=True)

# Plot
plt.figure()
plt.plot(df2024['month'],df2024['temperature'], 'kx', label='2024')
plt.xlabel('month')
plt.ylabel('Temperature (C)')
plt.title('Hourly temperatures, by month, for year 2024')
plt.grid()
plt.legend()

### Getting data for another year
In the cell below, your task is to duplicate the cell labeled "Loading (and plotting) time series of hourly temperature data, using Pandas" above, with modifications for the year 1977 (i.e., load, display, and plot).

In [ ]:
# Your code here


### Your turn
In the cell below, do some quality control on the 1977 dataframe you just made (get rid of bad data, and re-plot).

In [ ]:
# Your code here


### Plotting hourly temperatures on the same graph
You have probably already noticed that it's a bit difficult to compare two datasets unless you graph them together. In the cell below, we plot the 1977 and 2024 hourly temperature data on the same graph (still by month), using the black/blue coding we used above, and the label/legend method.

In [ ]:
plt.figure()
plt.plot(df2024['month'], df2024['temperature'], 'kx', label='2024')
plt.plot(df1977['month'], df1977['temperature'], 'b+', label='1977')
plt.xlabel('month')
plt.ylabel('Temperature (C)')
plt.title('Hourly temperatures, by month, for years 1977 and 2024')
plt.grid()
plt.legend()

### Pause for analysis
Take a moment to examine the plot you just made, and use the cell below to record a few observations about the seasonal variation it reveals. 

1. Back in 1977, which two months seem to have been the hottest?
1. Back in 1977, which two month seem to have been the coldest?
1. Although it's not considered "climate" unless one is averaging over (ideally) 30 years, sometimes we look at shorter time periods anyway, because that's the data we have. If you had to choose, which season would  seem to have warmed the more, winter or summer?

YOUR ANSWER HERE

### Focus on March
The cell below shows how to extract and display hourly temperatures belonging to a particular month of the year (March).

In [ ]:
# Specify March as the month we want to focus on
month_of_interest = 3

# Extract data belonging to that month
df2024_month_of_interest = df2024[df2024['month'] == month_of_interest]
display(df2024_month_of_interest) 

# Plot the temperature as a function of day
plt.figure()
plt.plot(df2024_month_of_interest['day'], df2024_month_of_interest['temperature'], 'kx')
plt.title('Hourly temperatures, by day, for month '+str(month_of_interest)+', year 2024')
plt.xlabel('day')
plt.ylabel('Temperature (C)')
plt.grid()

### Your turn
Repeat what we just did, but for 1977. Let's stick with the '+' and 'blue' representation we used before.

In [ ]:
# Your code here


### Probability densities of hourly temperatures
In the foregoing, you might have noticed that it's a little hard to infer trends from visual inspection of a time series. To do that, a useful statistical strategy is to create probability densities of temperature. The cell below uses numpy's "histogram" function to make a probability density for the 2024 dataset.

In [ ]:
# Get the histogram for the modern dataset
h2024_month_of_interest, e2024_month_of_interest = np.histogram(df2024_month_of_interest['temperature'],density=True)

# Check on some array lengths
print(np.size(h2024_month_of_interest))
print(np.size(e2024_month_of_interest))
print(np.size(e2024_month_of_interest[1:]))

# Plot the histogram 
plt.figure()
plt.plot(e2024_month_of_interest[1:],h2024_month_of_interest,'k')
plt.title('Hourly temperature probability density for month '+str(month_of_interest)+', year 2024')
plt.xlabel('Temperature (C)')
plt.grid()

### Pause for analysis

It's worth pausing for a moment on the meaning of the x-axis in figures like the above. If we see that the peak in a probability density occurs at an x-value of (say) -28, it means that the most probable hourly temperature that month fell within a certain range of -28 degrees C. That range is called a *bin*, and is decided automatically by np.histogram -- in this case, it has decided that each bin should be about three degrees in width. 

You might have noticed a strange notation here too: Why are we specifying e2024_month_of_interest[1:]? The short story is, a set of 10 *bins* requires that we specify 11 *edges* (AKA _bin boundaries_). The "[1:]" leaves off the first edge.

### Your turn
In the cell below, calculate the analogous probability density you just got, but for the 1977 dataset. 

In [ ]:
# Your code here


### Combining plots
In the cell below, we plot both probability densities on the same graph. 

In [ ]:
plt.figure()
plt.plot(e1977_month_of_interest[1:],h1977_month_of_interest,'b',label='1977')
plt.plot(e2024_month_of_interest[1:],h2024_month_of_interest,'k',label='2024')
plt.title('Hourly temperature probability density for month '+str(month_of_interest))
plt.xlabel('Temperature (C)')
plt.grid()
plt.legend()

### Pause for analysis
Use the cell below to comment on what these data are telling us about climate change in Utqiakvik between the years 1977 and 2024. Key ideas include:
1. The *most-probable temperature* (this is the temperature corresponding to the peak in the distribution)
1. The *spread* (or *range*) of temperatures

YOUR ANSWER HERE

### Combining dataframes from multiple years
Below is an example of how to merge dataframes from multiple years. This will come in handy for building confidence in statistical inferences we can draw from these data.

In [ ]:
# Modern data: Here we load four more years of data (2020-2023) as separate dataframes
df2020 = pd.read_csv('https://gml.noaa.gov/aftp/data/meteorology/in-situ/brw/met_brw_insitu_1_obop_hour_2020.txt', 
                        delimiter=r"\s+",header=None, 
                        usecols=[0,1,2,3,5,6,9], 
                        names=['station','year','month','day','winddirection','windspeed','temperature']) 

df2021 = pd.read_csv('https://gml.noaa.gov/aftp/data/meteorology/in-situ/brw/met_brw_insitu_1_obop_hour_2021.txt', 
                        delimiter=r"\s+",header=None, 
                        usecols=[0,1,2,3,5,6,9], 
                        names=['station','year','month','day','winddirection','windspeed','temperature']) 

df2022 = pd.read_csv('https://gml.noaa.gov/aftp/data/meteorology/in-situ/brw/met_brw_insitu_1_obop_hour_2022.txt', 
                        delimiter=r"\s+",header=None, 
                        usecols=[0,1,2,3,5,6,9], 
                        names=['station','year','month','day','winddirection','windspeed','temperature']) 

df2023 = pd.read_csv('https://gml.noaa.gov/aftp/data/meteorology/in-situ/brw/met_brw_insitu_1_obop_hour_2023.txt', 
                        delimiter=r"\s+",header=None, 
                        usecols=[0,1,2,3,5,6,9], 
                        names=['station','year','month','day','winddirection','windspeed','temperature']) 

df2025 = pd.read_csv('https://gml.noaa.gov/aftp/data/meteorology/in-situ/brw/met_brw_insitu_1_obop_hour_2025.txt', 
                        delimiter=r"\s+",header=None, 
                        usecols=[0,1,2,3,5,6,9], 
                        names=['station','year','month','day','winddirection','windspeed','temperature']) 

# Now we join them with 2024
df2020s = pd.concat( [df2020, df2021, df2022, df2023, df2024, df2025])

# Quality control
badindices = df2020s[ df2020s['temperature'] == -999.9 ].index
print('I am dropping this many missing data points: ', len(badindices))
df2020s.drop(badindices,inplace=True)

# Making the histogram on the entire dataset
df2020s_month_of_interest = df2020s[df2020s['month'] == month_of_interest]
h2020s_month_of_interest, e2020s_month_of_interest = np.histogram(df2020s_month_of_interest['temperature'],density=True)

plt.figure()
plt.plot(e2020s_month_of_interest[1:],h2020s_month_of_interest,'k',label='2020s')
plt.title('Hourly temperature probability densities for month '+str(month_of_interest))
plt.xlabel('Temperature (C)')
plt.grid()
plt.legend()

# Your turn
Do the same for 1978 and 1979, and concatenate with 1977. Call the resulting dataframe "df1970s".

In [ ]:
# Your code here


### Comparisons
In the cell below, plot these two probability densities (for the 1970s and the 2020s) on the same graph, using the label/legend annotation method.

In [ ]:
# Your code here


### Pause for analysis
Hopefully, the multi-year probability densities for the month of March that you just got are less noisy (less spiky), and therefore easier to identify trends in, compared to the single-year probability densities you got before. In the space below, comment on any trends you're seeing, focusing (as before) on changes in the most probable temperature and in the spread of temperatures.

YOUR ANSWER HERE

### Refresh/save/validate
Validate, close, submit, and log out.